https://www.census.gov/data/developers/data-sets/qwi.html?

In [1]:
import pandas as pd
import requests
from dotenv import load_dotenv
load_dotenv()
import os

In [ ]:
# By Agegroup-Industry

response = requests.get("https://api.census.gov/data/timeseries/qwi/sa", 
                        params={
                        "get": "Emp,HirN,Sep,industry", #Dropped Payroll since it is returning NA for the sa endpoint ---
                        "for": "state:08",
                        "time": "from 2020-Q1 to 2026-Q4",
                        "ind_level" : "S",
                        "sex": "0",
                        "agegrp": ["A01", "A02", "A03", "A04", "A05", "A06", "A07", "A08"],
                        "key": os.environ["CENSUS_API_KEY"]
                        }
                    )

response.raise_for_status()

In [ ]:
url = "https://api.census.gov/data/2024/acs/acs5"

params = {
    "get": "NAME,group(B01001)",
    "for": "metropolitan statistical area/micropolitan statistical area (or part):*",
    "in": "state:08",
    "key": os.environ["CENSUS_API_KEY"]
}

population = pd.DataFrame(requests.get(url, params=params).json())
population.columns = population.iloc[0]
population = population.iloc[1:].reset_index(drop=True)

In [ ]:
# By Agegroup-Industry-Metro


response2 = requests.get("https://api.census.gov/data/timeseries/qwi/sa", 
                        params={
                        "get": "Emp,HirN,Sep,industry", #Dropped Payroll since it is returning NA for the sa endpoint ---
                        "for": "metropolitan statistical area/micropolitan statistical area:*",
                        "in": "state:08",
                        "time": "from 2020-Q1 to 2026-Q4",
                        "ind_level" : "S",
                        "sex": "0",
                        "agegrp": ["A01", "A02", "A03", "A04", "A05", "A06", "A07", "A08"],
                        "key": os.environ["CENSUS_API_KEY"]
                        }
                    )


In [39]:
df = pd.DataFrame(response.json()[1:], columns=response.json()[0])


df["date"] = pd.PeriodIndex(
                           df["time"], 
                           freq="Q"
                            ).to_timestamp()

In [40]:
# Codebook

vars_qwi = pd.read_csv("../Data/Census Codebooks/variables_qwi.csv")
vars_industry = pd.read_csv("../Data/Census Codebooks/label_industry.csv")
naics_codes = pd.read_csv("../Data/Census Codebooks/NAICS.csv")
agegrp_codes = pd.read_csv("../Data/Census Codebooks/label_agegrp.csv")

In [41]:
df = df.apply( #apply -> takes each column from df as a series
    lambda col: col #input is column, output is column
        if col.name in ['time',
                        'date',
                        'industry',
                        'ind_level',
                        'agegrp']
        else pd.to_numeric(col, errors='coerce'),
        axis=0
            )

In [42]:
# Adding Industry Names to DF with codes only

df = df.merge(
    naics_codes[['NAICS','Sector']],
    left_on = 'industry',
    right_on = 'NAICS',
    how = 'left',
    validate = 'm:1'
)

In [43]:
# Adding age group ranges

df = df.merge(
    agegrp_codes[['agegrp','label']],
    left_on = 'agegrp',
    right_on = 'agegrp',
    how = 'left',
    validate = 'm:1'
)

df = df.rename(columns={'label':'age'})

In [ ]:

11	Agriculture, Forestry, Fishing and Hunting
21	Mining, Quarrying, Oil and Gas Extraction
22	Utilities
23	Construction



31-33	Manufacturing
42	Wholesale Trade
44-45	Retail Trade
48-49	Transportation and Warehousing



51	Information
54	Professional, Scientific, and Technical Services



52	Finance and Insurance
53	Real Estate and Rental and Leasing
55	Management of Companies and Enterprises



56	Administrative and Support Services
61	Educational Services



62	Health Care and Social Assistance
92	Public Administration



71	Arts, Entertainment, and Recreation
72	Accommodation and Food Services
81	Other Services


In [54]:
import altair as alt


plot = alt.Chart(df).mark_area().encode(
    x='date:T',
    y='sum(Emp):Q',
    color="Sector:N",
    tooltip = ['Sector:N', 'sum(Emp):Q']
)

In [55]:
plot.properties(width=750, height=400).interactive() + plot.mark_line()

alt.LayerChart(...)

In [58]:
plot = alt.Chart(df).mark_line().encode(
        x='date:T',
        y='Emp',
        color="age:N",
        column=('Sector:N'),
        tooltip= ['age:N', 'sum(Emp):Q']
) 

In [61]:
plot.mark_point().interactive()

alt.Chart(...)

In [60]:
# Summary Stats and General Trends by Age group for Colorado